# Gerenciador de Assets Visuais — SEGAPE/MEC

Ferramenta interativa para adicionar imagens ao repositório e gerar fórmulas para o Looker Studio.

Compatível com Jupyter Notebook, Google Colab e BigQuery Notebook.

## 1. Configuração inicial

Execute a célula abaixo para instalar as dependências necessárias.

In [ ]:
%pip install unidecode requests --quiet

## 2. Selecionar e enviar arquivo

Execute a célula abaixo para selecionar a imagem que deseja adicionar ao repositório.

- **Google Colab**: abrirá um diálogo de upload de arquivo
- **Jupyter com widgets**: exibirá um botão de seleção de arquivo
- **BigQuery Notebook / Terminal**: digite o caminho do arquivo (local ou gs://)

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
from pathlib import Path

try:
    from unidecode import unidecode
except ImportError:
    def unidecode(texto):
        return texto

# --- Detecção de ambiente ---
AMBIENTE = "terminal"

try:
    from google.colab import files as colab_files
    AMBIENTE = "colab"
except ImportError:
    try:
        import ipywidgets as widgets
        from IPython.display import display, HTML, clear_output
        AMBIENTE = "jupyter_widgets"
    except ImportError:
        try:
            from IPython import get_ipython
            if get_ipython() is not None:
                AMBIENTE = "notebook"
        except ImportError:
            pass

# --- Suporte a GCS (opcional) ---
baixar_de_gcs = None
try:
    from google.cloud import storage as gcs_storage

    def _baixar_de_gcs(uri_gcs):
        """Baixa arquivo do Google Cloud Storage. Aceita URIs gs://bucket/caminho."""
        partes = uri_gcs.replace("gs://", "").split("/", 1)
        bucket_nome = partes[0]
        blob_caminho = partes[1]
        nome = Path(blob_caminho).name
        cliente = gcs_storage.Client()
        bucket = cliente.bucket(bucket_nome)
        blob = bucket.blob(blob_caminho)
        conteudo = blob.download_as_bytes()
        return nome, conteudo

    baixar_de_gcs = _baixar_de_gcs
except ImportError:
    pass

# --- Constantes ---
REPOSITORIO = "SEGAPE/imagens_ux_ui"
BRANCH = "main"
URL_CDN_BASE = "https://cdn.jsdelivr.net/gh/{repo}@{branch}"
GITHUB_API_BASE = "https://api.github.com"
TIPOS_VALIDOS = ["logo", "icon"]

# Tentar importar funções do gerenciador local (quando disponível)
_IMPORTOU_GERENCIADOR = False
try:
    from scripts.gerenciador import sanitizar_nome, montar_nome_arquivo, gerar_url_cdn
    _IMPORTOU_GERENCIADOR = True
except ImportError:
    pass

# --- Funções auxiliares ---
if not _IMPORTOU_GERENCIADOR:
    def sanitizar_nome(texto):
        """Remove acentos, converte para snake_case limpo."""
        resultado = unidecode(texto)
        resultado = resultado.lower()
        resultado = re.sub(r"[\s\-\.]+", "_", resultado)
        resultado = re.sub(r"[^a-z0-9_]", "_", resultado)
        resultado = re.sub(r"_+", "_", resultado)
        return resultado.strip("_")

    def montar_nome_arquivo(programa, tipo, variante, extensao):
        """Monta nome padronizado: {programa}_{tipo}[_{variante}].{extensao}"""
        programa_san = sanitizar_nome(programa)
        tipo_san = sanitizar_nome(tipo)
        extensao = extensao.lstrip(".")
        if variante:
            variante_san = sanitizar_nome(variante)
            return f"{programa_san}_{tipo_san}_{variante_san}.{extensao}"
        return f"{programa_san}_{tipo_san}.{extensao}"

    def gerar_url_cdn(caminho_relativo):
        """Retorna URL jsDelivr CDN para o caminho."""
        base = URL_CDN_BASE.format(repo=REPOSITORIO, branch=BRANCH)
        return f"{base}/{caminho_relativo}"

# --- Estado global ---
arquivos_enviados = []

def carregar_arquivo_local(caminho_str):
    """Carrega arquivo de caminho local ou GCS."""
    caminho_str = caminho_str.strip()

    if caminho_str.startswith("gs://") and baixar_de_gcs is not None:
        nome, conteudo = baixar_de_gcs(caminho_str)
        extensao = Path(nome).suffix.lower() or ".png"
        return {"nome_original": nome, "conteudo": conteudo, "extensao": extensao}

    if caminho_str.startswith("gs://") and baixar_de_gcs is None:
        print("google-cloud-storage não disponível. Instale com: %pip install google-cloud-storage")
        return None

    caminho = Path(caminho_str)
    if not caminho.exists():
        print(f"Arquivo não encontrado: {caminho_str}")
        return None

    with open(caminho, "rb") as f:
        conteudo = f.read()
    extensao = caminho.suffix.lower() or ".png"
    return {"nome_original": caminho.name, "conteudo": conteudo, "extensao": extensao}

def upload_arquivo():
    """Gerencia o upload conforme o ambiente detectado."""
    global arquivos_enviados

    if AMBIENTE == "colab":
        resultado = colab_files.upload()
        for nome, conteudo in resultado.items():
            extensao = Path(nome).suffix.lower() or ".png"
            arquivos_enviados.append({
                "nome_original": nome,
                "conteudo": conteudo,
                "extensao": extensao,
            })
            print(f"Arquivo recebido: {nome}")

    elif AMBIENTE == "jupyter_widgets":
        uploader = widgets.FileUpload(
            accept="image/*",
            multiple=True,
            description="Selecionar imagem",
        )
        botao = widgets.Button(
            description="Confirmar seleção",
            button_style="success",
        )
        saida = widgets.Output()

        def ao_confirmar(_):
            with saida:
                clear_output()
                if not uploader.value:
                    print("Nenhum arquivo selecionado.")
                    return
                for item in uploader.value:
                    nome = item.name if hasattr(item, "name") else item["name"]
                    conteudo = item.content if hasattr(item, "content") else item["content"]
                    extensao = Path(nome).suffix.lower() or ".png"
                    arquivos_enviados.append({
                        "nome_original": nome,
                        "conteudo": bytes(conteudo),
                        "extensao": extensao,
                    })
                    print(f"Arquivo recebido: {nome}")

        botao.on_click(ao_confirmar)
        display(widgets.VBox([uploader, botao, saida]))

    else:
        print("Digite o caminho do arquivo de imagem.")
        if baixar_de_gcs is not None:
            print("Aceita caminhos locais ou URIs gs://bucket/caminho/arquivo.png")
        print("Digite 'fim' para encerrar a seleção.\n")

        while True:
            caminho = input("Caminho do arquivo (ou 'fim'): ").strip()
            if caminho.lower() == "fim":
                break
            resultado = carregar_arquivo_local(caminho)
            if resultado:
                arquivos_enviados.append(resultado)
                print(f"  Arquivo recebido: {resultado['nome_original']}")

print(f"Ambiente detectado: {AMBIENTE}")
upload_arquivo()

## 3. Classificar os arquivos

Preencha o programa, tipo e variante para cada arquivo enviado.

In [ ]:
assets_classificados = []

for i, arq in enumerate(arquivos_enviados):
    print(f"\n--- Arquivo {i+1}/{len(arquivos_enviados)}: {arq['nome_original']} ---")
    programa = input("Programa (ex: cnca, painel_ministro): ").strip()
    tipo = ""
    while tipo not in TIPOS_VALIDOS:
        tipo = input("Tipo (logo/icon): ").strip().lower()
        if tipo not in TIPOS_VALIDOS:
            print(f"  Tipo inválido. Escolha entre: {', '.join(TIPOS_VALIDOS)}")
    variante = input("Variante (opcional, ex: bronze, escuro -- deixe vazio se não houver): ").strip() or None

    nome_final = montar_nome_arquivo(programa, tipo, variante, arq["extensao"])
    programa_san = sanitizar_nome(programa)
    caminho_repo = f"assets/{programa_san}/{nome_final}"
    url_cdn = gerar_url_cdn(caminho_repo)

    assets_classificados.append({
        "nome_original": arq["nome_original"],
        "conteudo": arq["conteudo"],
        "programa": programa_san,
        "tipo": tipo,
        "variante": sanitizar_nome(variante) if variante else None,
        "nome_final": nome_final,
        "caminho_repo": caminho_repo,
        "url_cdn": url_cdn,
    })

    print(f"  Nome padronizado: {caminho_repo}")
    print(f"  URL CDN: {url_cdn}")

print(f"\n{len(assets_classificados)} arquivo(s) classificado(s).")

## 4. Enviar para o GitHub

Informe seu token de acesso pessoal (PAT) do GitHub com permissão de escrita no repositório.

Para criar um token: [github.com/settings/tokens](https://github.com/settings/tokens) -- "Generate new token (classic)" -- marcar "repo" -- copiar o token.

In [ ]:
import getpass
import requests

token = getpass.getpass("Token GitHub (PAT com permissão repo): ").strip()

if not token:
    print("Token não informado. Upload cancelado.")
else:
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
    }

    enviados = 0
    for asset in assets_classificados:
        caminho = asset["caminho_repo"]
        url_api = f"{GITHUB_API_BASE}/repos/{REPOSITORIO}/contents/{caminho}"

        resp_get = requests.get(url_api, headers=headers)
        sha = None
        if resp_get.status_code == 200:
            sha = resp_get.json().get("sha")

        conteudo_b64 = base64.b64encode(asset["conteudo"]).decode("utf-8")
        dados = {
            "message": f"feat: adiciona {asset['nome_final']}",
            "content": conteudo_b64,
            "branch": BRANCH,
        }
        if sha:
            dados["sha"] = sha

        resp_put = requests.put(url_api, headers=headers, json=dados)
        if resp_put.status_code in (200, 201):
            print(f"Enviado: {caminho}")
            enviados += 1
        else:
            print(f"Erro ao enviar {caminho}: {resp_put.status_code} -- {resp_put.json().get('message', '')}")

    print(f"\n{enviados}/{len(assets_classificados)} arquivo(s) enviado(s) com sucesso.")

## 5. Gerar fórmula para o Looker Studio

Preencha as informações abaixo para gerar a fórmula CASE/WHEN pronta para colar no Looker Studio.
Você pode usar os arquivos enviados nesta sessão ou assets existentes no catálogo do repositório.

In [ ]:
import requests

print("=== Gerador de fórmula CASE/WHEN ===")
print()

nome_campo = input("Nome do campo condicional (ex: selo, tipo_escola): ").strip()
if not nome_campo:
    print("Nome do campo obrigatório.")
else:
    linhas = []
    print(f"\nDefina as linhas da fórmula.")
    print(f"Para cada linha, informe o valor de '{nome_campo}' e a URL da imagem.")
    print(f"Digite 'fim' no valor para encerrar.\n")

    if assets_classificados:
        print("--- Assets enviados nesta sessão ---")
        for i, asset in enumerate(assets_classificados):
            ident = f"{asset['programa']}/{asset['tipo']}"
            if asset.get("variante"):
                ident += f"/{asset['variante']}"
            print(f"  [{i+1}] {ident} -- {asset['url_cdn']}")
        print()

    assets_catalogo = []
    catalogo_path = Path("catalogo.json")
    if catalogo_path.exists():
        with open(catalogo_path, "r", encoding="utf-8") as f:
            assets_catalogo = json.load(f)
    else:
        url_catalogo = gerar_url_cdn("catalogo.json")
        try:
            resp = requests.get(url_catalogo, timeout=10)
            if resp.status_code == 200:
                assets_catalogo = resp.json()
        except Exception:
            pass

    if assets_catalogo:
        print("--- Assets disponíveis no catálogo ---")
        for i, asset in enumerate(assets_catalogo):
            ident = f"{asset.get('programa', '?')}/{asset.get('tipo', '?')}"
            var = asset.get("variante")
            if var:
                ident += f"/{var}"
            print(f"  [c{i+1}] {ident} -- {asset.get('url_cdn', '')}")
        print()

    while True:
        valor = input(f"Valor de '{nome_campo}' (ou 'fim'): ").strip()
        if valor.lower() == "fim":
            break

        url = input(f"  URL ou número do asset ([1], [c1], etc.): ").strip()

        if url.startswith("c") and url[1:].isdigit():
            idx = int(url[1:]) - 1
            if 0 <= idx < len(assets_catalogo):
                url = assets_catalogo[idx].get("url_cdn", "")
            else:
                print("  Índice inválido.")
                continue
        elif url.isdigit():
            idx = int(url) - 1
            if 0 <= idx < len(assets_classificados):
                url = assets_classificados[idx]["url_cdn"]
            else:
                print("  Índice inválido.")
                continue

        linhas.append((valor, url))
        print(f"  Adicionado: WHEN {nome_campo} = \"{valor}\" THEN \"{url}\"")

    if linhas:
        print("\n=== Fórmula gerada ===\n")
        formula = "CASE\n"
        for valor, url in linhas:
            formula += f'  WHEN {nome_campo} = "{valor}" THEN "{url}"\n'
        formula += f"  WHEN {nome_campo} IS NULL THEN NULL\n"
        formula += "  ELSE NULL\n"
        formula += "END"
        print(formula)
        print("\nCopie a fórmula acima e cole no campo calculado do Looker Studio.")
    else:
        print("Nenhuma linha adicionada.")

## Pronto!

A fórmula gerada pode ser colada diretamente no campo calculado do Looker Studio.
Lembre-se de configurar o tipo do campo como **Imagem** e usar o componente **Tabela** para exibir.